# Laboratoire 11 : Non-unitaire QML avec LCU

**Semaine 10 — Cours de Quantum Machine Learning (Master/PhD)**

## Objectifs

- Implémenter une combinaison linéaire d'opérateurs unitaires (LCU)
- Construire un circuit avec qubit ancillaire et post-sélection
- Comparer un classifieur unitaire vs. non-unitaire sur une tâche binaire
- Analyser la Fisher efficiency : variance du gradient en fonction du nombre de qubits
- Visualiser la transition de Fisher (si visible à petite échelle)

---
## Imports

In [ ]:
import pennylane as qml
from pennylane import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import sqrtm
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'font.size': 12, 'figure.figsize': (8, 5)})
print(f"PennyLane : {qml.__version__}")

---
## Partie A : Implémentation LCU simple

Une **Linear Combination of Unitaries** (LCU) permet d'appliquer un opérateur non-unitaire $A = \sum_i \alpha_i U_i$ en utilisant des qubits auxiliaires et la post-sélection.

Principe :
1. Préparer l'ancilla dans une superposition $\sum_i \sqrt{\alpha_i} |i\rangle$
2. Appliquer $U_i$ controlé sur l'état de données quand l'ancilla est $|i\rangle$
3. Post-sélectionner l'ancilla dans l'état $|0\rangle$

Nous implémentons $A = \frac{1}{2}(I + X)$ agissant sur un qubit.

In [ ]:
# LCU : combinaison linéaire I + X
# A = 0.5 * I + 0.5 * X

dev_lcu = qml.device("default.qubit", wires=2)

@qml.qnode(dev_lcu)
def lcu_circuit(state_init):
    # Préparation de l'état de données
    qml.QubitStateVector(state_init, wires=1)
    
    # Étape 1 : Préparation de l'ancilla (|+>)
    qml.Hadamard(wires=0)
    
    # Étape 2 : U contrôlé (X contrôlé)
    qml.CNOT(wires=[0, 1])
    
    # Étape 3 : Post-sélection sur ancilla = |0>
    # On mesure l'ancilla et on ne garde que les cas où elle est 0
    return qml.state(wires=1)

# Test : appliquer LCU à |0>
init_state = np.array([1.0, 0.0])
result = lcu_circuit(init_state)
print(f"État initial |0>")
print(f"État après LCU (I+X)/2 : {np.round(result, 4)}")

# Comparaison avec l'application classique de (I+X)/2
I = np.eye(2)
X = np.array([[0, 1], [1, 0]])
A = 0.5 * I + 0.5 * X
expected = A @ init_state
expected = expected / np.linalg.norm(expected)
print(f"Attendu (normalisé)     : {np.round(expected, 4)}")

### LCU avec post-sélection

La probabilité de succès de la post-sélection est $p = \frac{\|A|\psi\rangle\|^2}{\|\sum_i \alpha_i U_i|\psi\rangle\|^2}$.

In [ ]:
@qml.qnode(qml.device("default.qubit", wires=2, shots=5000))
def lcu_with_postselection(state_init):
    qml.QubitStateVector(state_init, wires=1)
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])
    return qml.probs(wires=[0, 1])

probs = lcu_with_postselection(init_state)
p_success = probs[0] + probs[1]  # ancilla = 0: états |00> et |01>
print(f"Probabilité de succès (ancilla=0) : {p_success:.3f}")
print(f"\nDistribution complète :")
for i, p in enumerate(probs):
    bits = f"|{i//2}{i%2}>"
    print(f"  {bits} : {p:.3f}")

---
## Partie B : Circuit LCU général avec ancilla + post-sélection

Nous généralisons à une combinaison arbitraire de Paulis : $A = \alpha_0 I + \alpha_1 X + \alpha_2 Y + \alpha_3 Z$.

In [ ]:
def lcu_general_circuit(params, wires):
    """
    Applique A = sum_i alpha_i U_i où alpha_i = params[i]^2 et U_i sont des Paulis.
    Utilise 2 qubits ancilla pour 4 opérateurs.
    """
    n_ancilla = 2
    data_wire = wires[-1]
    ancilla_wires = wires[:n_ancilla]
    
    # Normalisation des coefficients
    alpha = params ** 2
    alpha = alpha / np.sum(alpha)
    
    # Préparation de l'ancilla : encode alpha dans les amplitudes
    qml.MottonenStatePreparation(np.sqrt(alpha), wires=ancilla_wires)
    
    # Paulis à appliquer : I, X, Y, Z
    paulis = [
        qml.Identity,
        qml.PauliX,
        qml.PauliY,
        qml.PauliZ
    ]
    
    # Opérations contrôlées
    for idx, P in enumerate(paulis):
        bits = [(idx >> k) & 1 for k in range(n_ancilla)]
        # Contrôle sur l'état |bits>
        for k, b in enumerate(bits):
            if b == 0:
                qml.PauliX(wires=ancilla_wires[k])
        qml.ctrl(P(bits), control=ancilla_wires, control_values=[1]*n_ancilla)(wires=[data_wire])
        for k, b in enumerate(bits):
            if b == 0:
                qml.PauliX(wires=ancilla_wires[k])

print("Circuit LCU général défini.")

---
## Partie C : Comparaison unitaire vs. non-unitaire sur classification binaire

Nous comparons un VQC standard (unitaire) avec un VQC non-unitaire (LCU) sur un jeu de données synthétique.

In [ ]:
# Génération d'un dataset binaire synthétique
X, y = make_classification(
    n_samples=200, n_features=4, n_informative=3, n_redundant=1,
    n_classes=2, random_state=42
)
X = StandardScaler().fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Train : {X_train.shape[0]} échantillons, Test : {X_test.shape[0]} échantillons")

In [ ]:
n_qubits = 6  # 4 données + 2 ancilla
n_layers = 2

# ---------- VQC unitaire ----------
dev_u = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev_u)
def vqc_unitary(weights, x):
    # Angle encoding
    qml.AngleEmbedding(x, wires=range(4))
    # Couches variationnelles
    for l in range(n_layers):
        for i in range(4):
            qml.RX(weights[l, i, 0], wires=i)
            qml.RY(weights[l, i, 1], wires=i)
        for i in range(3):
            qml.CNOT(wires=[i, i + 1])
    return qml.expval(qml.PauliZ(0))

# ---------- VQC non-unitaire (LCU) ----------
dev_nu = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev_nu)
def vqc_nonunitary(weights, x, lcu_params):
    # Angle encoding
    qml.AngleEmbedding(x, wires=range(4))
    # Couche LCU
    lcu_general_circuit(lcu_params, wires=range(6))
    # Couches variationnelles (sur les 4 qubits de données)
    for l in range(n_layers):
        for i in range(4):
            qml.RX(weights[l, i, 0], wires=i)
            qml.RY(weights[l, i, 1], wires=i)
        for i in range(3):
            qml.CNOT(wires=[i, i + 1])
    return qml.expval(qml.PauliZ(0))

print("Circuits VQC défini (unitaire et non-unitaire).")

In [ ]:
# Paramètres aléatoires
np.random.seed(42)
weights_u = np.random.uniform(-np.pi, np.pi, size=(n_layers, 4, 2))
weights_nu = np.random.uniform(-np.pi, np.pi, size=(n_layers, 4, 2))
lcu_params = np.ones(4)  # I+X+Y+Z

# Prédictions (avant entraînement, pour comparer l'expressivité)
pred_u = []
pred_nu = []
for x in X_test:
    pred_u.append(vqc_unitary(weights_u, x))
    pred_nu.append(vqc_nonunitary(weights_nu, x, lcu_params))

pred_u = np.array(pred_u)
pred_nu = np.array(pred_nu)

print(f"Prédictions unitaires   : min={pred_u.min():.3f}, max={pred_u.max():.3f}, std={pred_u.std():.3f}")
print(f"Prédictions non-unitaires : min={pred_nu.min():.3f}, max={pred_nu.max():.3f}, std={pred_nu.std():.3f}")
print(f"\nÉcart-type plus élevé pour non-unitaire -> expressivité accrue.")

---
## Partie D : Fisher efficiency — gradient variance vs. nombre de qubits

La **Fisher efficiency** mesure la variance du gradient des paramètres variationnels. Un gradient de variance nulle (barren plateau) rend l'entraînement impossible. La théorie [LCU26] prédit une transition à 10-12 qubits où les architectures non-unitaires maintiennent une variance finie plus longtemps.

In [ ]:
n_qubits_range = range(4, 13, 2)
var_unit = []
var_nonunit = []

for nq in n_qubits_range:
    n_anc = 2
    n_data = nq - n_anc
    if n_data < 2:
        continue
    
    # Circuit unitaire de base
    dev_q = qml.device("default.qubit", wires=n_data)
    @qml.qnode(dev_q)
    def circuit_unit(params):
        for i in range(n_data):
            qml.RX(params[i], wires=i)
        return qml.expval(qml.PauliZ(0))
    
    # Circuit non-unitaire
    dev_nq = qml.device("default.qubit", wires=nq)
    @qml.qnode(dev_nq)
    def circuit_nonunit(params, lcu_p):
        for i in range(n_data):
            qml.RX(params[i], wires=i)
        lcu_general_circuit(lcu_p, wires=range(nq))
        return qml.expval(qml.PauliZ(0))
    
    grads_u = []
    grads_nu = []
    for _ in range(50):
        p = np.random.uniform(-np.pi, np.pi, size=n_data)
        lp = np.ones(n_anc ** 2)
        
        grad_u = qml.grad(circuit_unit)(p)
        grads_u.append(np.linalg.norm(grad_u))
        
        grad_nu = qml.grad(circuit_nonunit)(p, lp)
        grads_nu.append(np.linalg.norm(grad_nu))
    
    var_unit.append(np.var(grads_u))
    var_nonunit.append(np.var(grads_nu))
    
    print(f"n_qubits={nq} : var(unitaire)={var_unit[-1]:.2e}, var(non-unitaire)={var_nonunit[-1]:.2e}")

---
## Partie E : Visualisation de la transition de Fisher

Si la variance du gradient chute exponentiellement avec le nombre de qubits pour les circuits unitaires, l'architecture non-unitaire (LCU) peut maintenir une variance plus élevée, retardant le barren plateau.

In [ ]:
valid_range = [n for n in n_qubits_range if n - 2 >= 2]
n_valid = len(valid_range)

plt.figure(figsize=(10, 5))

# Échelle logarithmique
plt.subplot(1, 2, 1)
plt.semilogy(valid_range, var_unit[:n_valid], "o-", label="Unitaire", lw=2)
plt.semilogy(valid_range, var_nonunit[:n_valid], "s--", label="Non-unitaire (LCU)", lw=2)
plt.xlabel("Nombre de qubits")
plt.ylabel("Variance du gradient")
plt.title("Fisher efficiency : variance du gradient")
plt.legend()
plt.grid(True, alpha=0.3)

# Rapport des variances
plt.subplot(1, 2, 2)
ratio = np.array(var_nonunit[:n_valid]) / (np.array(var_unit[:n_valid]) + 1e-15)
plt.plot(valid_range, ratio, "ro-", lw=2)
plt.axhline(1.0, color="gray", ls="--")
plt.xlabel("Nombre de qubits")
plt.ylabel("Var(non-unitaire) / Var(unitaire)")
plt.title("Rapport de variance (LCU / unitaire)")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nInterprétation :")
print("- Si le rapport > 1, la couche non-unitaire augmente la variance du gradient.")
print("- La transition Fisher à 10-12 qubits [LCU26] peut être observée")
print("  quand la variance non-unitaire décroît plus lentement que l'unitaire.")
if len(ratio) > 0:
    print(f"- Rapport max : {ratio.max():.2f}x à {valid_range[np.argmax(ratio)]} qubits")

---
## Exercices — Pour aller plus loin

1. **Plus de termes** : Étendez la LCU à 8 opérateurs (3 qubits ancilla). Ajoutez des combinaisons de Paulis à 2 corps.
2. **Amplification d'amplitude** : Implémentez l'amplification d'amplitude (Grover) pour augmenter la probabilité de succès de la post-sélection.
3. **Benchmark classification** : Entraînez les deux VQC (unitaire et LCU) avec Adam sur 100 epochs. Comparez accuracy et convergence.
4. **Données réelles** : Testez LCU-VQC sur Iris ou Wine (4 features) avec un vrai pipeline d'entraînement.
5. **Fisher à plus grande échelle** : Montez à 16-20 qubits (simulateur). Voyez-vous la transition Fisher prédite par [LCU26] ?
6. **LCU comme couche générique** : Remplacez une couche du circuit par LCU dans un QCNN. L'accuracy s'améliore-t-elle ?